# Schemas and Data Types: Explicit StructTypes, Nested Types & Schema Enforcement

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 02_schemas_and_data_types.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How to define explicit StructType schemas, parse complex nested types (ArrayType, MapType, StructType), and enforce data contracts.
- **Why does this topic exist?**: Relying on inferSchema causes runtime performance hits and pipeline failures when dirty records arrive.
- **What problem does it solve?**: InferSchema triggers an extra full dataset scan and silently breaks when data types vary across batch files.
- **Where is it used in real-world Data Engineering?**: Enterprise data ingestion frameworks enforce strict schemas to reject corrupted records into dead-letter queues.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrame fundamentals |
| **Required depth** | Working knowledge: can define and validate schemas, including nested types |
| **Data Engineer** | Define explicit schemas, validate incoming data, and handle schema drift safely. |
| **Senior Data Engineer** | Design ingestion contracts and choose between enforcement, evolution, and quarantine. |
| **Spark specialist** | Reason about type coercion, ANSI mode, and nested structure representation. |
| **Next topic** | `03_dataframe_and_spark_sql/03_dataframe_transformations.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Schema inference scans data and guesses types. That is convenient for exploration but dangerous for
production: a leading zero can become an integer, a null can change a type, and a new column can silently
disappear. An explicit schema is an ingestion contract; schema evolution is a controlled change to that contract.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **StructType** | A table schema | Ordered collection of StructFields |
| **StructField** | One column definition | Name, data type, nullable flag, metadata |
| **Inference** | Guess types from data | Extra scan and heuristic risk |
| **Enforcement** | Reject or coerce bad data | Explicit schema and validation |
| **Nested type** | Struct/array/map inside a column | Common in JSON and event data |
| **Schema evolution** | Change the contract over time | Add columns, merge schemas, migrate readers |

### How it works, step by step

1. The reader parses records and maps them to the schema.
2. The analyzer checks column names and types against the plan.
3. A cast or coercion may happen at read time or in an expression.
4. ANSI mode can raise errors for invalid casts instead of returning null.
5. Nested JSON is represented as structs, arrays, and maps.
6. Schema evolution rules determine whether old files remain readable.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Schema Architecture & Data Types

```
StructType (Schema Definition)
  |-- StructField("order_id", IntegerType, nullable=False)
  |-- StructField("customer_id", IntegerType, nullable=True)
  |-- StructField("items", ArrayType(StructType(...)), nullable=True)  <-- Complex Nested Type
  |-- StructField("metadata", MapType(StringType, StringType))        <-- Key-Value Map Type
```

### Why Explicit Schemas Matter in Production:
1. **Zero Scan Overhead**: Spark skips the pre-scan required by `inferSchema=True`.
2. **Type Safety**: Enforces strict contracts (e.g. `order_id` is never accidentally parsed as String).
3. **Deterministic Evolution**: Allows controlled handling of new or missing columns.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Schemas_DataTypes_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Defining Explicit Schemas for Retail Orders

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType, TimestampType
)

orders_schema = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=False),
    StructField("product_id", IntegerType(), nullable=False),
    StructField("order_ts", TimestampType(), nullable=False),
    StructField("quantity", IntegerType(), nullable=False),
    StructField("unit_price", DoubleType(), nullable=False),
    StructField("discount", DoubleType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("channel", StringType(), nullable=True)
])

# Read with explicit schema enforcement
orders_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv("/opt/spark-data/retail/orders.csv")
)

orders_df.printSchema()
print(f"Total verified orders count: {orders_df.count()}")
orders_df.show(3)

## 4. Complex Nested Types: Parsing JSON (`/opt/spark-data/retail/order_events.json`)

Now let us parse deeply nested JSON containing nested StructTypes and ArrayTypes.

In [ ]:
events_raw = spark.read.text("/opt/spark-data/retail/order_events.json")

# Define nested schema matching JSON structure
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_time", StringType()),
    StructField("order", StructType([
        StructField("order_id", IntegerType()),
        StructField("status", StringType()),
        StructField("channel", StringType()),
        StructField("items", ArrayType(StructType([
            StructField("product_id", IntegerType()),
            StructField("product_name", StringType()),
            StructField("category", StringType()),
            StructField("quantity", IntegerType()),
            StructField("unit_price", DoubleType())
        ])))
    ])),
    StructField("customer", StructType([
        StructField("customer_id", IntegerType()),
        StructField("segment", StringType()),
        StructField("region", StringType()),
        StructField("country", StringType())
    ]))
])

# Parse JSON strings with explicit schema
events_df = (
    events_raw
    .select(F.from_json(F.col("value"), event_schema).alias("data"))
    .select("data.*")
)

events_df.printSchema()
events_df.select("event_id", "order.order_id", "customer.country").show(3)

---

## Experiment: Exploding Nested Arrays into Relational Rows

- **1. What are we testing?**: Use F.explode() on order.items to flatten nested array elements into individual row records.
- **2. What do we expect?**: Each item in the items array will become a distinct row retaining parent order attributes.

In [ ]:
flattened = (
    events_df
    .select(
        F.col("event_id"),
        F.col("order.order_id").alias("order_id"),
        F.col("customer.customer_id").alias("customer_id"),
        F.explode(F.col("order.items")).alias("item")
    )
    .select(
        "event_id", "order_id", "customer_id",
        "item.product_name", "item.quantity", "item.unit_price"
    )
)

flattened.show(5)
print(f"Original events: {events_df.count()} -> Flattened rows: {flattened.count()}")

**3. What happened?**: Array items were converted to separate rows; row count expanded proportionally.  
**4. Why did it happen?**: explode() unrolls an ArrayType column, replicating parent fields for every nested element.  
**5. Practical Engineering Lesson**: F.explode() is the primary tool in Spark SQL for normalizing semi-structured document payloads.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Ingestion contract for a payments feed

- **Industry**: Banking
- **Scenario**: A bank ingests daily payment files from several vendors.
- **Problem**: Vendor schemas drift, amounts arrive as strings, and a silently inferred type can corrupt reconciliation.
- **Why this concept is relevant**: An explicit schema plus validation rejects or quarantines malformed records before they reach the ledger.
- **Implementation idea**: Define StructType, read with `mode=PERMISSIVE`, add a `_corrupt_record` column, validate counts and ranges, and quarantine bad rows.
- **Expected result**: A clean, typed payments table and an auditable reject set.
- **Alternative**: A schema registry plus a streaming validation service.
- **Trade-offs**: Spark validation is batch-friendly and transparent; a registry/service centralizes contracts across many producers.

### Use Case 2: Nested event data from a mobile app

- **Industry**: E-commerce
- **Scenario**: A mobile app sends JSON events with nested `order.items` arrays.
- **Problem**: Flattening the JSON by hand is error-prone, and the nested structure changes between app versions.
- **Why this concept is relevant**: Spark's nested types let the team query structs and arrays directly and evolve the schema with `mergeSchema`.
- **Implementation idea**: Read JSON, inspect `printSchema()`, explode items for product analytics, and keep the raw nested column for replay.
- **Expected result**: Both nested and flattened views without losing the original event.
- **Alternative**: A document store or a JSON-native warehouse.
- **Trade-offs**: Spark handles nested analytics at scale; a document store is better for operational lookups.

---

## Business Value

- Prevents silent type corruption in financial and operational data.
- Makes data contracts explicit and testable.
- Reduces downstream failures and manual reconciliation.
- Supports safer schema evolution and backward compatibility.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use explicit schemas for production ingestion.
- Validate row counts, null rates, ranges, and distinct keys before publishing.
- Use `mode=PERMISSIVE` with `_corrupt_record` for quarantine workflows.
- Document and test schema evolution; do not assume `mergeSchema` is free.
- Be aware of ANSI mode: invalid casts may fail the job.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Inference vs explicit schema

**Steps**

1. Read `orders.csv` with `inferSchema=True` and print the schema.
2. Read it again with an explicit StructType and compare.
3. Introduce a malformed value and observe permissive vs failfast behavior.

**What to observe**: Inference adds a scan and can choose a wider type; explicit schemas make failures predictable.

**Healthy result**: The explicit schema matches the documented contract.

**Problematic result**: A vendor changes a column and inference silently changes the type.

**Engineering takeaway**: Treat schemas as code and review changes.

---

## Trade-offs and Alternatives

### Advantages

- Strong typing and validation
- Nested data support
- Controlled evolution

### Limitations

- More upfront work
- Evolution can break old readers
- Type coercion can still surprise

### When to use it

- Production ingestion
- JSON/event data
- Data contracts

### When not to use it

- Guessing schemas on critical data
- Uncontrolled mergeSchema in every job

### Alternatives inside Spark

- `from_json`/`to_json`
- `mergeSchema`
- `_corrupt_record`

### Alternatives outside Spark

- Schema registry
- Avro/Protobuf
- dbt tests

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Extract Nested Struct Field
**Problem Statement**: Extract customer segment and country directly using dot-notation into new top-level columns.  
**Input / Context**: events_df

In [ ]:
# Exercise 1: Extract nested fields

### Exercise 2 (Intermediate): Handle Corrupted Record Mode with PERMISSIVE
**Problem Statement**: Load /opt/spark-data/quality/orders_dirty.csv using PERMISSIVE mode and columnNameOfCorruptRecord.  
**Input / Context**: /opt/spark-data/quality/orders_dirty.csv

In [ ]:
# Exercise 2: Corrupted record tracking

### Exercise 3 (Advanced): Construct Schema with MapType
**Problem Statement**: Create a DataFrame with a MapType(StringType, DoubleType) and query a specific map key.  
**Input / Context**: F.create_map()

In [ ]:
# Exercise 3: MapType usage

---

## Production Debugging Exercise

### The leading-zero bug

**Scenario**: A customer ID column contains values like `00123`; inference reads it as an integer and drops the leading zeros.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the inferred type.
2. Read the column as a string and validate the format.
3. Decide whether the identifier is numeric or a code.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Extract Nested Struct Field):
```python
res = events_df.select(
    F.col("order.order_id").alias("order_id"),
    F.col("customer.segment").alias("cust_segment"),
    F.col("customer.country").alias("cust_country")
)
res.show(3)
```

#### Solution 2 (Handle Corrupted Record Mode with PERMISSIVE):
```python
dirty_schema = orders_schema.add(StructField("_corrupt_record", StringType(), True))
dirty_df = (
    spark.read
    .option("header", True)
    .option("mode","PERMISSIVE")
    .option("columnNameOfCorruptRecord","_corrupt_record")
    .schema(dirty_schema)
    .csv("/opt/spark-data/quality/orders_dirty.csv")
)
print("Dirty records count with nulls or format errors:")
dirty_df.filter(F.col("_corrupt_record").isNotNull()).select("_corrupt_record").show(3, truncate=False)
```

#### Solution 3 (Construct Schema with MapType):
```python
map_df = spark.createDataFrame([
    (1, {"tax": 12.5,"discount": 5.0}),
    (2, {"tax": 8.0,"discount": 0.0})
], ["id","financials"])
map_df.printSchema()
map_df.select("id", F.col("financials")["tax"].alias("tax_amount")).show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Explicit Schemas**: Mandatory in production; eliminates scan overhead and enforces contract safety.
- **Nested Data**: Spark natively supports StructType, ArrayType, and MapType.
- **explode()**: Converts ArrayType elements into individual relational rows.

### Essential Technical Terms
- `StructType`: Collection of StructFields representing a schema.
- `StructField`: Single column definition (name, type, nullable).
- `explode`: Function that flattens an array into multiple rows.

### Common Engineering Mistakes
- **inferSchema=True in Production**: Slows down ingestion and risks schema changes when dirty data arrives.
- **Ignoring Nullability**: Setting nullable=False does not actively reject nulls in CSV; it is a metadata hint.

### Interview Questions on This Topic
- **Q: Why use explicit schemas over inferSchema?**
  - *A*: Explicit schemas avoid an extra full data scan and make parsing deterministic, but they do not guarantee that incoming values are semantically valid.
- **Q: How do you query nested StructType fields?**
  - *A*: Using dot notation (e.g. col('customer.address.city')).


### More Interview Questions

- **Q: Why is `inferSchema=True` risky in production?**
  - *A*: It requires an extra pass over the data and guesses types from a sample, which can change when the data changes.
- **Q: What is the difference between schema enforcement and schema evolution?**
  - *A*: Enforcement rejects or coerces data that does not match the contract; evolution deliberately changes the contract while preserving compatibility.

### What to Study Next
Open 03_dataframe_and_spark_sql/03_dataframe_transformations.ipynb for the complete transformation catalog.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')